In [ ]:
#  Guiseley is a real-world water distribution system covering 6 km², serving a population of 4,500, supplied by gravity, and containing 
# 451 junctions and 497 pipes.

# To test the Principle of Parsimony, I decided to test this principle on the Tema East-Tema West Regional WDN

In [ ]:
import wntr
import numpy as np
from scipy.optimize import differential_evolution

inp_file = "C:/Users/user/Downloads/python_projects/TMR_Base+Model_calibrated_for_07_04_26_V10.inp"
wn =wntr.network.WaterNetworkModel(inp_file)
# wn.options.hydraulic.headloss = 'H-W'


import pandas as pd

# target nodes: TC_1_4418 and TC_2_4418
# 24-HOUR DIURNAL TELEMETRY LOGGER DATA (in meters)

# ==============================================================================

observed_pressures = {
    'TC_1_4418': [
        15.220637, 17.10460409, 18.59128316, 18.90257772, 
        18.18471273, 15.59810384, 11.08694458, 10.55104065, 
        10.19119008, 10.50633494, 11.01376343, 11.40501912, 
        11.93593852, 12.67667135, 13.10154724, 13.2795105, 
        13.41372681, 12.94197591, 12.40741984, 9.28208287, 
        3.811131795, 4.273506165, 4.868871053, 5.4772288
    ],
    'TC_2_4448': [
        17.54563904, 18.18030294, 18.31240336, 19.68271383,
        18.87998962, 14.82647705, 7.684315999, 5.88277181,
        5.719530741, 5.442078908, 5.153703054, 4.510505676,
        5.178686778, 5.586537679, 5.058906555, 5.468691508,
        5.328464508, 4.360720317, 4.53483963, 4.701740265,
        4.990070343, 5.386750539, 6.229733785, 8.61824798
    ]
}

# # Verify data structures match
# for logger, timesteps in observed_pressures.items():
#     print(f"Logger: {logger} | Loaded Timesteps: {len(timesteps)} hours of field data.")


# guiseley step (Principle of Parsimony)

pipe_groups = {}
pipe_to_group_map = {}

for name, pipe in wn.pipes():

    # let's reduce the sample size from 62
    # in wntr, pipe diametres are in metres. so we need to convert to mm
    raw_diam = pipe.diameter * 1000 
    normalized_diam = int(round(raw_diam / 50) * 50)

    if normalized_diam == 0:
        normalized_diam = 25
    
                          
    # group_id = f"Dia_{int(pipe.diameter * 1000)} mm"
    group_id = f"Dia_{normalized_diam}mm"

    if group_id not in pipe_groups:
        pipe_groups[group_id]= []
    pipe_groups[group_id].append(name)
    # pipe_to_group_map[name] = group_id

group_names = list(pipe_groups.keys())
num_groups = len(group_names)

print(f"Parsimony used here: search space is {num_groups}")
# print(group_names)
 
# Note: if the optimization algorithm outputs a strange roughness value 
# or causes a hydraulic crash at a specific pipe, you can type 
# pipe_to_group_map['Pipe_X'] in Jupyter. It will instantly tell you 
# which diameter class is causing the error without making you search through every list.

def calibration_obj(roughness_vector):

    # here we take a vector of roughness valves and run a simulation
    # and calculate the SSE.
    
    # lets assign the current roughness to all the pipes in every group
    for idx, group_name in enumerate(group_names):
        # By default, enumerate() starts counting from 0. It returns a tuple 
        # containing the index and the value, which you can easily unpack in a for loop

        current_roughness = roughness_vector[idx]
        for pipe_name in pipe_groups[group_name]:
            wn.get_link(pipe_name).roughness = current_roughness

    # Run the EPANET Simulator
    try:
        sim = wntr.sim.EpanetSimulator(wn)
        results = sim.run_sim()
    except Exception:
        return 1e10 # return a huge error if the model crashes
    
        # calculate the SSE

    sse = 0.0
    for node_name, obs_values in observed_pressures.items():
        try:
            sim_values = results.node['pressure'].loc[:,node_name].values[:24]
            sse += np.sum((obs_values - sim_values)**2)
        except KeyError:
            print(f"Warning: Node {node_name} is not found in results matrox")
            return 1e10
    return sse # this was previously missing, so the coude

bounds = [(0.0015, 2.5)] * num_groups
# bounds  = [(60, 140)] * num_groups
# result = differential_evolution(calibration_obj, bounds, 
#                                 strategy = 'best1bin', maxiter=10, popsize= 5,
#                                tol = 0.05, workers = -1)
# previously used maxiter = 50, popsize = 10 (it kept running after 20 minutes)
# also included workers = - 1 (to spread the processing across all CPUs)
# also added tol = 0.05 to stop the calibration early if model stops improving 

print("Running the differential evolution alg...")

result = differential_evolution(
    calibration_obj, 
    bounds, 
    strategy='best1bin', 
    maxiter=3,        # Slashed to 3 generations for a rapid proof-of-concept
    popsize=3,        # Small population size for quick execution
    tol=0.1,          # High tolerance for fast convergence
    workers=1, # CRITICAL: Forces Python to run sequentially on 1 core to stop freezing
    polish=False,
    disp=True
)


print("\n--- Calibration Run Complete ---")
print(f"Minimum Sum of Squared Errors (SSE) Achieved: {result.fun:.4f}")
for idx, group_name in enumerate(group_names):
    print(f"Optimized Darcy-Weisbach Roughness for {group_name}: {result.x[idx]:.2f}")

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\wntr\network\options.py:420: UserWarning: Changing the headloss formula from H-W to D-W will not change the units of the roughness coefficient.
  warnings.warn('Changing the headloss formula from ' +
C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\wntr\epanet\io.py:2082: UserWarning: Not all curves were used in "C:/Users/user/Downloads/python_projects/TMR_Base+Model_calibrated_for_07_04_26_V10.inp"; added with type None, units conversion left to user
  warnings.warn('Not all curves were used in "{}"; added with type None, units conversion left to user'.format(self.wn.name))


Parsimony used here: search space is 19
Running the differential evolution alg...


In [ ]:
# import wntr
# import numpy as np
# from scipy.optimize import differential_evolution

In [ ]:
# inp_file = "C:/Users/user/Downloads/python_projects/TMR_Base+Model_calibrated_for_07_04_26_V10.inp"

In [ ]:
# wn =wntr.network.WaterNetworkModel(inp_file)

In [ ]:
# import pandas as pd

In [ ]:
# this is just to keep record of how to pull attributes using the "pipe.attribute"

pipe_registry = []
for name, pipe in wn.pipes():
    pipe_registry.append({
        'pipe_id':name,
        'diameter': pipe.diameter,
        'roughness': pipe.roughness,
        'length': pipe.length,
        'status': pipe.initial_status
            
    })

df_pipes = pd.DataFrame(pipe_registry)
df_pipes.head()
    

In [ ]:
# # target nodes: TC_1_4418 and TC_2_4418
# # 24-HOUR DIURNAL TELEMETRY LOGGER DATA (in meters)

# # ==============================================================================

# observed_pressures = {
#     'TC_1_4418': [
#         15.220637, 17.10460409, 18.59128316, 18.90257772, 
#         18.18471273, 15.59810384, 11.08694458, 10.55104065, 
#         10.19119008, 10.50633494, 11.01376343, 11.40501912, 
#         11.93593852, 12.67667135, 13.10154724, 13.2795105, 
#         13.41372681, 12.94197591, 12.40741984, 9.28208287, 
#         3.811131795, 4.273506165, 4.868871053, 5.4772288
#     ],
#     'TC_2_4448': [
#         17.54563904, 18.18030294, 18.31240336, 19.68271383,
#         18.87998962, 14.82647705, 7.684315999, 5.88277181,
#         5.719530741, 5.442078908, 5.153703054, 4.510505676,
#         5.178686778, 5.586537679, 5.058906555, 5.468691508,
#         5.328464508, 4.360720317, 4.53483963, 4.701740265,
#         4.990070343, 5.386750539, 6.229733785, 8.61824798
#     ]
# }

# # # Verify data structures match
# # for logger, timesteps in observed_pressures.items():
# #     print(f"Logger: {logger} | Loaded Timesteps: {len(timesteps)} hours of field data.")


In [ ]:
# # guiseley step (Principle of Parsimony)

# pipe_groups = {}
# pipe_to_group_map = {}

# for name, pipe in wn.pipes():
#     group_id = f"Dia_{int(pipe.diameter * 1000)} mm"

#     if group_id not in pipe_groups:
#         pipe_groups[group_id]= []
#     pipe_groups[group_id].append(name)
#     # pipe_to_group_map[name] = group_id

# group_names = list(pipe_groups.keys())
# num_groups = len(group_names)

# print(f"Parsimony used here: search space is {num_groups}")
# print(group_names)
 
# # Note: if the optimization algorithm outputs a strange roughness value 
# # or causes a hydraulic crash at a specific pipe, you can type 
# # pipe_to_group_map['Pipe_X'] in Jupyter. It will instantly tell you 
# # which diameter class is causing the error without making you search through every list.

In [ ]:
# here, the search is 62 as compared to more than 12,000 pipes



In [ ]:
# def calibration_obj(roughness_vector):

#     # here we take a vector of roughness valves and run a simulation
#     # and calculate the SSE.
    
#     # lets assign the current roughness to all the pipes in every group
#     for idx, group_name in enumerate(group_names):
#         # By default, enumerate() starts counting from 0. It returns a tuple 
#         # containing the index and the value, which you can easily unpack in a for loop

#         current_roughness = roughness_vector[idx]
#         for pipe_name in pipe_groups[group_name]:
#             wn.get_link(pipe_name).roughness = current_roughness

#     # Run the EPANET Simulator
#     try:
#         sim = wntr.sim.EpanetSimulator(wn)
#         results = sim.run_sim()
#     except Exception:
#         return 1e10 # return a huge error if the model crashes
    
#         # calculate the SSE

#     sse = 0.0
#     for node_name, obs_values in observed_pressures.items():
#         try:
#             sim_values = results.node['pressure'].loc[:,node_name].values[:24]
#             sse += np.sum((obs_values - sim_values)**2)
#         except KeyError:
#             print(f"Warning: Node {node_name} is not found in results matrox")
#             return 1e10
#     return sse # this was previously missing, so the coude
    

In [ ]:
# bounds  = [(60, 140)] * num_groups
# # result = differential_evolution(calibration_obj, bounds, 
# #                                 strategy = 'best1bin', maxiter=10, popsize= 5,
# #                                tol = 0.05, workers = -1)
# # previously used maxiter = 50, popsize = 10 (it kept running after 20 minutes)
# # also included workers = - 1 (to spread the processing across all CPUs)
# # also added tol = 0.05 to stop the calibration early if model stops improving 

# print("Switching to Single-Core Accelerated Track to prevent deadlocks...")

# result = differential_evolution(
#     calibration_obj, 
#     bounds, 
#     strategy='best1bin', 
#     maxiter=3,        # Slashed to 3 generations for a rapid proof-of-concept
#     popsize=3,        # Small population size for quick execution
#     tol=0.1,          # High tolerance for fast convergence
#     workers=1         # CRITICAL: Forces Python to run sequentially on 1 core to stop freezing
# )

In [ ]:
# print("\n--- Calibration Run Complete ---")
# print(f"Minimum Sum of Squared Errors (SSE) Achieved: {result.fun:.4f}")
# for idx, group_name in enumerate(group_names):
#     print(f"Optimized Hazen-Williams Roughness for {group_name}: {result.x[idx]:.2f}")

In [ ]:
# Print the global head loss formula used by the network model
print("Headloss Formula In Use:", wn.options.hydraulic.headloss)
